In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\code\portfolio_optimization")

print(PROJECT_ROOT)

C:\code\portfolio_optimization


In [9]:
# 05a3-1. 데이터 경로 설정

KRX_STOCK_PANEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "clean"
    / "krx"
    / "stocks"
    / "krx_stock_panel_clean.parquet"
)

MEMBERSHIP_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "krx"
    / "universe"
    / "krx300_monthly_membership_snapshots.csv"
)

print(
    "stock panel exists:",
    KRX_STOCK_PANEL_PATH.exists()
)

print(
    "membership exists:",
    MEMBERSHIP_PATH.exists()
)

stock panel exists: True
membership exists: True


In [10]:
# 05a3-2. monthly membership 불러오기

monthly_membership = pd.read_csv(
    MEMBERSHIP_PATH,
    dtype={
        "ticker": str,
        "index_code": str
    }
)

monthly_membership["date"] = pd.to_datetime(
    monthly_membership["date"]
)

monthly_membership["ticker"] = (
    monthly_membership["ticker"]
    .astype(str)
    .str.strip()
)


print(
    "rows:",
    len(monthly_membership)
)

print(
    "snapshots:",
    monthly_membership["date"].nunique()
)

print(
    "start:",
    monthly_membership["date"].min()
)

print(
    "end:",
    monthly_membership["date"].max()
)

rows: 31232
snapshots: 104
start: 2018-02-28 00:00:00
end: 2026-09-15 00:00:00


In [11]:
# 05a3-3. weekly signal 날짜 생성
# parquet에서 date만 불러와 각 주 마지막 거래일 생성

import pyarrow.parquet as pq


date_table = pq.read_table(
    KRX_STOCK_PANEL_PATH,
    columns=[
        "date"
    ]
)

date_data = date_table.to_pandas()


trading_dates = (
    date_data["date"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)


weekly_signal_dates = (
    trading_dates
    .to_frame(
        name="signal_date"
    )
)


weekly_signal_dates["week"] = (
    weekly_signal_dates["signal_date"]
    .dt.to_period("W-FRI")
)


weekly_signal_dates = (
    weekly_signal_dates
    .groupby(
        "week",
        as_index=False
    )
    ["signal_date"]
    .max()
)


print(
    "trading dates:",
    len(trading_dates)
)

print(
    "weekly signals:",
    len(weekly_signal_dates)
)

weekly_signal_dates.head()

trading dates: 2112
weekly signals: 450


,week,signal_date
0,2018-02-03/2018-02-09,2018-02-09
1,2018-02-10/2018-02-16,2018-02-14
2,2018-02-17/2018-02-23,2018-02-23
3,2018-02-24/2018-03-02,2018-03-02
4,2018-03-03/2018-03-09,2018-03-09


backtest 규칙
signal = 마지막 거래일 close
execution = 다음 거래일 open

In [12]:
# 05a3-4. 다음 거래일 계산

next_trade_date = pd.DataFrame({
    "signal_date": trading_dates[:-1].values,
    "execution_date": trading_dates[1:].values
})


weekly_signal_dates = (
    weekly_signal_dates
    .merge(
        next_trade_date,
        on="signal_date",
        how="left"
    )
)


weekly_signal_dates.head()

,week,signal_date,execution_date
0,2018-02-03/2018-02-09,2018-02-09,2018-02-12
1,2018-02-10/2018-02-16,2018-02-14,2018-02-19
2,2018-02-17/2018-02-23,2018-02-23,2018-02-26
3,2018-02-24/2018-03-02,2018-03-02,2018-03-05
4,2018-03-03/2018-03-09,2018-03-09,2018-03-12


In [13]:
# 05a3-5. signal date와 이전 membership snapshot 연결

snapshot_dates = (
    monthly_membership[
        ["date"]
    ]
    .drop_duplicates()
    .sort_values("date")
    .rename(
        columns={
            "date": "membership_date"
        }
    )
)


weekly_signal_dates = pd.merge_asof( #as of: 그 시점 기준 가장 최근에 알려진 값
    weekly_signal_dates.sort_values(
        "signal_date"
    ),
    snapshot_dates,
    left_on="signal_date",
    right_on="membership_date",
    direction="backward",
    allow_exact_matches=False #과거 snapshot만 사용, look-ahead bias 방지 위함
)


weekly_signal_dates.head(10)

,week,signal_date,execution_date,membership_date
0,2018-02-03/2018-02-09,2018-02-09,2018-02-12,NaT
1,2018-02-10/2018-02-16,2018-02-14,2018-02-19,NaT
2,2018-02-17/2018-02-23,2018-02-23,2018-02-26,NaT
3,2018-02-24/2018-03-02,2018-03-02,2018-03-05,2018-02-28
4,2018-03-03/2018-03-09,2018-03-09,2018-03-12,2018-02-28
5,2018-03-10/2018-03-16,2018-03-16,2018-03-19,2018-02-28
6,2018-03-17/2018-03-23,2018-03-23,2018-03-26,2018-02-28
7,2018-03-24/2018-03-30,2018-03-30,2018-04-02,2018-02-28
8,2018-03-31/2018-04-06,2018-04-06,2018-04-09,2018-03-30
9,2018-04-07/2018-04-13,2018-04-13,2018-04-16,2018-03-30


In [14]:
# 05a3-6. membership 없는 signal 확인

missing_membership = (
    weekly_signal_dates[
        "membership_date"
    ]
    .isna()
)


print(
    "missing signals:",
    missing_membership.sum()
)

weekly_signal_dates[
    missing_membership
].head(20)

missing signals: 3


,week,signal_date,execution_date,membership_date
0,2018-02-03/2018-02-09,2018-02-09,2018-02-12,NaT
1,2018-02-10/2018-02-16,2018-02-14,2018-02-19,NaT
2,2018-02-17/2018-02-23,2018-02-23,2018-02-26,NaT


In [15]:
# 05a3-7. membership 사용 가능한 signal만 유지

weekly_signal_dates = (
    weekly_signal_dates[
        weekly_signal_dates[
            "membership_date"
        ]
        .notna()
    ]
    .copy()
)


print(
    "usable signals:",
    len(weekly_signal_dates)
)

print(
    "first signal:",
    weekly_signal_dates[
        "signal_date"
    ].min()
)

print(
    "last signal:",
    weekly_signal_dates[
        "signal_date"
    ].max()
)

usable signals: 447
first signal: 2018-03-02 00:00:00
last signal: 2026-09-15 00:00:00


signal_date
↓
membership_date
↓
그 snapshot의 KRX300 ticker

In [16]:
# 05a3-8. signal별 KRX300 membership 생성

weekly_membership = (
    weekly_signal_dates[
        [
            "signal_date",
            "execution_date",
            "membership_date"
        ]
    ]
    .merge(
        monthly_membership[
            [
                "date",
                "ticker"
            ]
        ],
        left_on="membership_date",
        right_on="date",
        how="left"
    )
    .drop(
        columns=[
            "date"
        ]
    )
)


print(
    "rows:",
    len(weekly_membership)
)

print(
    "signals:",
    weekly_membership[
        "signal_date"
    ]
    .nunique()
)

weekly_membership.head()

rows: 134238
signals: 447


,signal_date,execution_date,membership_date,ticker
0,2018-03-02,2018-03-05,2018-02-28,005930
1,2018-03-02,2018-03-05,2018-02-28,000660
2,2018-03-02,2018-03-05,2018-02-28,068270
3,2018-03-02,2018-03-05,2018-02-28,005380
4,2018-03-02,2018-03-05,2018-02-28,005490


In [17]:
# 05a3-9. signal별 membership 수 확인

members_per_signal = (
    weekly_membership
    .groupby(
        "signal_date"
    )
    ["ticker"]
    .nunique()
)


print(
    members_per_signal.describe()
)

members_per_signal.tail()

count    447.000000
mean     300.308725
std        1.763733
min      297.000000
25%      299.000000
50%      300.000000
75%      301.000000
max      305.000000
Name: ticker, dtype: float64


signal_date
2026-08-21    300
2026-08-28    300
2026-09-04    301
2026-09-11    301
2026-09-15    301
Name: ticker, dtype: int64

In [19]:
# 05a3-10. signal date stock 데이터 연결
# parquet에서 weekly signal 날짜만 불러옴

import pyarrow.parquet as pq


stock_columns = [
    "date",
    "ticker",
    "name",
    "market",
    "close",
    "volume",
    "trading_value",
    "market_cap",
    "is_no_trade",
    "is_invalid_ohlc"
]


signal_dates_list = (
    weekly_signal_dates["signal_date"]
    .dropna()
    .tolist()
)


stock_table = pq.read_table(
    KRX_STOCK_PANEL_PATH,
    columns=stock_columns,
    filters=[
        (
            "date",
            "in",
            signal_dates_list
        )
    ]
)


signal_stock_data = (
    stock_table
    .to_pandas()
    .rename(
        columns={
            "date": "signal_date"
        }
    )
)


print(
    "signal stock rows:",
    len(signal_stock_data)
)

print(
    "signal dates:",
    signal_stock_data["signal_date"].nunique()
)

signal stock rows: 1118954
signal dates: 447


In [20]:
# 05a3-10. weekly membership과 stock 데이터 연결

weekly_universe = (
    weekly_membership
    .merge(
        signal_stock_data,
        on=[
            "signal_date",
            "ticker"
        ],
        how="left",
        indicator=True
    )
)


print(
    weekly_universe[
        "_merge"
    ]
    .value_counts()
)

_merge
both          134234
left_only          4
right_only         0
Name: count, dtype: int64


전체 stock panel 5,278,578행
        ↓
weekly signal 날짜만 parquet에서 선택
        ↓
필요한 10개 컬럼만 로드
        ↓
weekly membership과 merge

In [21]:
# 05a3-11. 미매칭 종목 확인

unmatched = (
    weekly_universe[
        weekly_universe["_merge"] == "left_only"
    ]
    .copy()
)

print(
    "unmatched:",
    len(unmatched)
)

unmatched[
    [
        "signal_date",
        "execution_date",
        "membership_date",
        "ticker"
    ]
]

unmatched: 4


,signal_date,execution_date,membership_date,ticker
15205,2019-02-15,2019-02-18,2019-01-31,000030
15505,2019-02-22,2019-02-25,2019-01-31,000030
15805,2019-02-28,2019-03-04,2019-01-31,000030
80941,2023-04-28,2023-05-02,2023-03-31,008560


In [44]:
# 05a3-12. candidate universe 생성
# stock panel과 연결된 종목만 유지

candidate_universe = (
    weekly_universe[
        weekly_universe["_merge"] == "both"
    ]
    .drop(
        columns=[
            "_merge"
        ]
    )
    .copy()
)

print(
    "rows:",
    len(candidate_universe)
)

print(
    "signals:",
    candidate_universe["signal_date"].nunique()
)

print(
    candidate_universe
    .groupby("signal_date")
    ["ticker"]
    .nunique()
    .describe()
)

rows: 134234
signals: 447
count    447.000000
mean     300.299776
std        1.769086
min      297.000000
25%      299.000000
50%      300.000000
75%      301.000000
max      305.000000
Name: ticker, dtype: float64


lagged monthly krx300
        ↓
weekly signal
        ↓
실제 시장 데이터 존재
        ↓
candidate universe

In [23]:
# 05a3-13. universe 선정용 stock history 불러오기
# trading value: 거래대금

universe_columns = [
    "date",
    "ticker",
    "close",
    "volume",
    "trading_value",
    "market_cap",
    "is_no_trade",
    "is_invalid_ohlc"
]

history_table = pq.read_table(
    KRX_STOCK_PANEL_PATH,
    columns=universe_columns
)

stock_history = (
    history_table
    .to_pandas()
    .sort_values(
        [
            "ticker",
            "date"
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    stock_history.shape
)

(5278578, 8)


In [24]:
# 05a3-14. 20일 평균 거래대금 계산
# 과거와 signal 당일 정보만 사용

stock_history["trading_value_ma20"] = (
    stock_history
    .groupby(
        "ticker",
        sort=False
    )
    ["trading_value"]
    .rolling(
        window=20,
        min_periods=20
    )
    .mean()
    .reset_index(
        level=0,
        drop=True
    )
)

In [25]:
print(
    stock_history[
        "trading_value_ma20"
    ]
    .isna()
    .sum()
)

stock_history[
    [
        "date",
        "ticker",
        "trading_value",
        "trading_value_ma20"
    ]
].tail()

60457


,date,ticker,trading_value,trading_value_ma20
5278573,2026-09-09,950260,12825861305,NaN
5278574,2026-09-10,950260,27916516590,NaN
5278575,2026-09-11,950260,10869479095,NaN
5278576,2026-09-14,950260,7297436410,1.702555e+11
5278577,2026-09-15,950260,16920945085,1.342957e+11


In [41]:
# 05a3-13a. stock history 존재 확인

print(
    "stock_history exists:",
    "stock_history" in globals()
)

if "stock_history" in globals():

    print(
        "shape:",
        stock_history.shape
    )

stock_history exists: True
shape: (5278578, 10)


In [42]:
# 05a3-14a. 종목별 history 길이 계산

if "stock_history" not in globals():

    universe_columns = [
        "date",
        "ticker",
        "close",
        "volume",
        "trading_value",
        "market_cap",
        "is_no_trade",
        "is_invalid_ohlc"
    ]

    history_table = pq.read_table(
        KRX_STOCK_PANEL_PATH,
        columns=universe_columns
    )

    stock_history = (
        history_table
        .to_pandas()
        .sort_values(
            [
                "ticker",
                "date"
            ]
        )
        .reset_index(
            drop=True
        )
    )


stock_history["history_count"] = (
    stock_history
    .groupby(
        "ticker",
        sort=False
    )
    .cumcount()
    + 1
)


print(
    "shape:",
    stock_history.shape
)

print(
    stock_history[
        [
            "date",
            "ticker",
            "history_count"
        ]
    ].head()
)

shape: (5278578, 10)
        date  ticker  history_count
0 2018-02-05  000020              1
1 2018-02-06  000020              2
2 2018-02-07  000020              3
3 2018-02-08  000020              4
4 2018-02-09  000020              5


In [43]:
# 05a3-14b. 20일 평균 거래대금 확인

if "trading_value_ma20" not in stock_history.columns:

    stock_history["trading_value_ma20"] = (
        stock_history
        .groupby(
            "ticker",
            sort=False
        )
        ["trading_value"]
        .rolling(
            window=20,
            min_periods=20
        )
        .mean()
        .reset_index(
            level=0,
            drop=True
        )
    )


print(
    stock_history[
        [
            "trading_value_ma20",
            "history_count"
        ]
    ].notna()
    .sum()
)

trading_value_ma20    5218121
history_count         5278578
dtype: int64


In [45]:
# 05a3-15. liquidity와 history 연결

signal_liquidity = (
    stock_history[
        [
            "date",
            "ticker",
            "trading_value_ma20",
            "history_count"
        ]
    ]
    .rename(
        columns={
            "date": "signal_date"
        }
    )
)

candidate_universe = (
    candidate_universe
    .merge(
        signal_liquidity,
        on=[
            "signal_date",
            "ticker"
        ],
        how="left"
    )
)

stock panel 매칭
close > 0
signal 당일 거래 있음
ohlc 오류 아님
20일 평균 거래대금 존재
market cap > 0

In [46]:
# 05a3-16. investable candidate 조건 적용

eligible_universe = (
    candidate_universe[
        (candidate_universe["close"] > 0)
        &
        (candidate_universe["volume"] > 0)
        &
        (~candidate_universe["is_no_trade"])
        &
        (~candidate_universe["is_invalid_ohlc"])
        &
        (candidate_universe["market_cap"] > 0)
        &
        (candidate_universe["trading_value_ma20"].notna())
        &
        (candidate_universe["history_count"] >= 60)
    ]
    .copy()
)

eligible_count = (
    eligible_universe
    .groupby("signal_date")
    ["ticker"]
    .nunique()
)

print(
    eligible_count.describe()
)

print(
    "first signal:",
    eligible_universe["signal_date"].min()
)

count    438.000000
mean     299.011416
std        1.552250
min      295.000000
25%      298.000000
50%      299.000000
75%      300.000000
max      304.000000
Name: ticker, dtype: float64
first signal: 2018-05-04 00:00:00


krx300 candidate
        ↓
signal-day tradability filter
        ↓
20일 평균 거래대금 기준 rank
        ↓
top 50

In [28]:
# 05a3-17. eligible 종목이 없는 signal 확인

all_signal_dates = set(
    weekly_signal_dates["signal_date"]
)

eligible_signal_dates = set(
    eligible_universe["signal_date"]
)

missing_eligible_signals = sorted(
    all_signal_dates
    - eligible_signal_dates
)

print(
    "missing eligible signals:",
    missing_eligible_signals
)

missing eligible signals: [Timestamp('2018-03-02 00:00:00')]


In [29]:
# 05a3-18. 실제 universe 사용 시작일 확인

first_eligible_signal = (
    eligible_universe["signal_date"]
    .min()
)

print(
    "first eligible signal:",
    first_eligible_signal
)

print(
    "eligible signals:",
    eligible_universe[
        "signal_date"
    ].nunique()
)

first eligible signal: 2018-03-09 00:00:00
eligible signals: 446


lagged krx300
↓
거래 가능 여부 filter
↓
최근 20일 평균 거래대금 순위
↓
top 50

In [47]:
# 05a3-19. weekly investable universe 50개 선정
# ma20 거래대금 내림차순, 동률이면 시가총액 사용

eligible_universe = (
    eligible_universe
    .sort_values(
        [
            "signal_date",
            "trading_value_ma20",
            "market_cap",
            "ticker"
        ],
        ascending=[
            True,
            False,
            False,
            True
        ]
    )
    .copy()
)


eligible_universe["liquidity_rank"] = (
    eligible_universe
    .groupby(
        "signal_date"
    )
    .cumcount()
    + 1
)


investable_universe = (
    eligible_universe[
        eligible_universe[
            "liquidity_rank"
        ]
        <= 50
    ]
    .copy()
)

In [48]:
# 05a3-20. signal별 investable 종목 수 확인

investable_count = (
    investable_universe
    .groupby(
        "signal_date"
    )
    ["ticker"]
    .nunique()
)


print(
    investable_count.describe()
)

print(
    "\ncount distribution:"
)

print(
    investable_count
    .value_counts()
    .sort_index()
)

count    438.0
mean      50.0
std        0.0
min       50.0
25%       50.0
50%       50.0
75%       50.0
max       50.0
Name: ticker, dtype: float64

count distribution:
ticker
50    438
Name: count, dtype: int64


446개 weekly signal
×
50개 종목
=
22,300 rows

In [49]:
# 05a3-21. 첫 signal investable universe 확인
# sanity check = 결과가 상식적으로 말이 되는지 확인하는 간단한 검증

first_signal = (
    investable_universe[
        "signal_date"
    ]
    .min()
)


first_universe = (
    investable_universe[
        investable_universe[
            "signal_date"
        ]
        == first_signal
    ]
    [
        [
            "signal_date",
            "ticker",
            "name",
            "market",
            "trading_value_ma20",
            "market_cap",
            "liquidity_rank"
        ]
    ]
    .sort_values(
        "liquidity_rank"
    )
)


print(
    "signal:",
    first_signal
)

first_universe.head(20)

signal: 2018-05-04 00:00:00


,signal_date,ticker,name,market,trading_value_ma20,market_cap,liquidity_rank
2741,2018-05-04,005930,삼성전자,KOSPI,6.539449e+11,3.331630e+14,1
2742,2018-05-04,000660,SK하이닉스,KOSPI,3.467539e+11,6.042420e+13,2
2744,2018-05-04,068270,셀트리온,KOSPI,3.208275e+11,3.127709e+13,3
2746,2018-05-04,207940,삼성바이오로직스,KOSPI,2.611792e+11,2.378632e+13,4
2785,2018-05-04,000720,현대건설,KOSPI,2.021760e+11,7.338345e+12,5
2793,2018-05-04,215600,신라젠,KOSDAQ,1.862561e+11,5.224768e+12,6
2809,2018-05-04,028300,에이치엘비,KOSDAQ,1.616320e+11,4.475433e+12,7
2775,2018-05-04,009150,삼성전기,KOSPI,1.400515e+11,8.589775e+12,8
2747,2018-05-04,028260,삼성물산,KOSPI,1.341318e+11,2.437517e+13,9
2854,2018-05-04,064350,현대로템,KOSPI,1.298103e+11,2.703000e+12,10


In [50]:
# 05a3-22. weekly universe 변경량 확인

universe_sets = (
    investable_universe
    .groupby(
        "signal_date"
    )
    ["ticker"]
    .apply(set)
    .sort_index()
)


universe_changes = []

previous_date = None
previous_set = None


for signal_date, current_set in universe_sets.items():

    if previous_set is not None:

        added = (
            current_set
            - previous_set
        )

        removed = (
            previous_set
            - current_set
        )

        universe_changes.append({
            "signal_date": signal_date,
            "previous_date": previous_date,
            "added": len(added),
            "removed": len(removed)
        })

    previous_date = signal_date
    previous_set = current_set


universe_changes = pd.DataFrame(
    universe_changes
)


print(
    universe_changes[
        [
            "added",
            "removed"
        ]
    ]
    .describe()
)

            added     removed
count  437.000000  437.000000
mean     3.446224    3.446224
std      1.559218    1.559218
min      0.000000    0.000000
25%      2.000000    2.000000
50%      3.000000    3.000000
75%      5.000000    5.000000
max      8.000000    8.000000


시장 변화
+
liquidity rank 변화
+
membership 변화

때문에 universe가 얼마나 자주 바뀌는지 볼 수 있음

In [51]:
# 05a3-23. weekly investable universe 저장

import pyarrow as pa

UNIVERSE_DIR = (
    PROJECT_ROOT
    / "data"
    / "clean"
    / "krx"
    / "universe"
)

UNIVERSE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


INVESTABLE_UNIVERSE_PATH = (
    UNIVERSE_DIR
    / "krx300_weekly_investable_50.parquet"
)


investable_table = pa.Table.from_pandas(
    investable_universe,
    preserve_index=False
)


pq.write_table(
    investable_table,
    INVESTABLE_UNIVERSE_PATH,
    compression="snappy"
)


print(
    "saved:",
    INVESTABLE_UNIVERSE_PATH
)

saved: C:\code\portfolio_optimization\data\clean\krx\universe\krx300_weekly_investable_50.parquet


pipeline\
전체 kospi + kosdaq 5,278,578 rows -> lagged monthly krx300 proxy -> weekly signal 447개 -> stock panel 매칭 -> tradability + ma20 filter -> warm-up 1개 제외 -> 446개 signal -> 20일 평균 거래대금 -> top 50 -> weekly investable universe

In [37]:
print(
    "stock_history" in globals()
)

True


In [39]:
# 05a3-13a. stock history 존재 확인

print(
    "stock_history exists:",
    "stock_history" in globals()
)

if "stock_history" in globals():

    print(
        "shape:",
        stock_history.shape
    )

stock_history exists: True
shape: (5278578, 10)
